In [2]:
# Import libraries and packages
import pandas as pd
import networkx as nx
from networkx.algorithms import bipartite
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from matplotlib.path import Path

# configuration and metadata

# The minimum amount of volume change (%) per week needed to draw an edge
THRESHOLD = 15.0

# Store event, date, type, and pew score (5 events total covered)
events = {
    "Japan\nDisaster":       {"date": "3/11/2011",  "type": "International Natural Disaster", "pew": 55},
    "Gas & Oil\nPrices":    {"date": "4/15/2011",  "type": "Financial", "pew": 53},
    "Bin Laden\nDeath":     {"date": "5/6/2011",   "type": "Geopolitical",    "pew": 50},
    "Government\nShutdown": {"date": "4/8/2011",   "type": "Domestic Politics", "pew": 47},
    "Midwest\nTorandoes":   {"date": "5/27/2011",  "type": "Domestic Natural Disaster", "pew": 45},
}

# The stocks and their respective sectors in the dataset
sectors = {
    "AA":   "Materials",        "AXP":  "Financials",
    "BA":   "Industrials",      "BAC":  "Financials",
    "CAT":  "Industrials",      "CSCO": "Technology",
    "CVX":  "Energy",           "DD":   "Materials",
    "DIS":  "Consumer Disc.",   "GE":   "Industrials",
    "HD":   "Consumer Disc.",   "HPQ":  "Technology",
    "IBM":  "Technology",       "INTC": "Technology",
    "JNJ":  "Healthcare",       "JPM":  "Financials",
    "KO":   "Consumer Staples", "KRFT": "Consumer Staples",
    "MCD":  "Consumer Staples", "MMM":  "Industrials",
    "MRK":  "Healthcare",       "MSFT": "Technology",
    "PFE":  "Healthcare",       "PG":   "Consumer Staples",
    "T":    "Telecom",          "TRV":  "Financials",
    "UTX":  "Industrials",      "VZ":   "Telecom",
    "WMT":  "Consumer Staples", "XOM":  "Energy",
}

# Sector visual color appearance mappings
sector_colors = {
    "Energy":           "#FF6B6B",
    "Financials":       "#4FC3F7",
    "Technology":       "#CE93D8",
    "Industrials":      "#FFB74D",
    "Materials":        "#A1887F",
    "Healthcare":       "#81C784",
    "Consumer Staples": "#4DD0E1",
    "Consumer Disc.":   "#FF8A65",
    "Telecom":          "#90A4AE",
}

# Event category color assignments used for mapping types to their pew metrics
event_type_colors = {
    "International Natural Disaster": "#4FC3F7",
    "Financial":                      "#FF9800",
    "Geopolitical":                   "#EF5350",
    "Domestic Politics":              "#66BB6A",
    "Domestic Natural Disaster":      "#CE93D8", # Fixed plural matching typo here
}

# General color options for the figures (stylistic)
BG_COLOR      = "white"
TEXT_COLOR    = "#111111"
SUBTEXT_COLOR = "#666677"

# Build network

# Loading the raw ticker transaction data
df = pd.read_csv("dow_jones_index.data")

# Starts empty network structure
B = nx.Graph()

# Add the event nodes and stock nodes into the bipartite setup
for name, meta in events.items():
    B.add_node(name, node_type="event", **meta)
for ticker, sector in sectors.items():
    B.add_node(ticker, node_type="stock", sector=sector)

# Loops through the stock data by week for each event, checks threshold, creates edge if met
edge_data = []
for event_name, meta in events.items():
    week = df[df["date"] == meta["date"]][["stock", "percent_change_volume_over_last_wk"]].dropna()
    for _, row in week.iterrows():
        vol = row["percent_change_volume_over_last_wk"]
        if abs(vol) >= THRESHOLD:
            B.add_edge(event_name, row["stock"], weight=abs(vol), raw=vol)
            edge_data.append({
                "event":           event_name.replace("\n", " "),
                "stock":           row["stock"],
                "sector":          sectors[row["stock"]],
                "volume_change_%": round(vol, 2),
            })

# Extracts distinct nodes for events and stocks for accurate metrics calculation
event_nodes = [n for n in B.nodes if B.nodes[n]["node_type"] == "event"]
stock_nodes = [n for n in B.nodes if B.nodes[n]["node_type"] == "stock"]

# Console summary


print("BIPARTITE NETWORK SUMMARY")

print(f"Threshold : ±{THRESHOLD}%")
print(f"Nodes     : {B.number_of_nodes()} ({len(events)} events + {len(sectors)} stocks)")
print(f"Edges     : {B.number_of_edges()}")
print()

for event_name in events:
    neighbors = sorted(B.neighbors(event_name))
    print(f"{'─' * 50}")
    print(f"Event : {event_name.replace(chr(10), ' ')}")
    print(f"  Reacting stocks ({len(neighbors)}): {', '.join(neighbors)}")
    for n in neighbors:
        raw   = B[event_name][n]["raw"]
        arrow = "▲" if raw > 0 else "▼"
        print(f"    {n:<6}  {arrow} {raw:+.1f}%   [{sectors[n]}]")

print()
edge_df = pd.DataFrame(edge_data)
print("Sector reaction counts across all events:")
print(edge_df.groupby("sector").size().sort_values(ascending=False).to_string())
print()

multi = {t: B.degree(t) for t in sectors if B.degree(t) > 1}
print("Bellwether stocks (reacted to more than one event):")
for ticker, deg in sorted(multi.items(), key=lambda x: -x[1]):
    print(f"  {ticker:<6}  {deg} events  [{sectors[ticker]}]")
print()

# Network metrics

print("NETWORK METRICS")


# Observes overall network density: max possible edges = n_events * n_stocks
max_possible = len(event_nodes) * len(stock_nodes)
density = B.number_of_edges() / max_possible
print(f"\nOverall network density : {density:.3f}")
print(f"  ({B.number_of_edges()} edges out of {max_possible} possible)")
print(f"  Interpretation: {density*100:.1f}% of all event-stock pairs showed a significant volume reaction")

# Observes per-event density to see how broadly each individual event spread
print("\nPer-event density (breadth of market impact):")
for ev in event_nodes:
    ev_edges   = B.degree(ev)
    ev_density = ev_edges / len(stock_nodes)
    label      = ev.replace("\n", " ")
    bar        = "█" * int(ev_density * 40)
    print(f"  {label:<22}  {ev_density:.3f}  {bar}  ({ev_edges} stocks)")

# Degree centrality calculation (stocks)
print("\nDegree centrality — stocks (top 10 most connected):")
stock_centrality = bipartite.degree_centrality(B, event_nodes)
top_stocks = sorted(
    [(n, c) for n, c in stock_centrality.items() if n in stock_nodes],
    key=lambda x: -x[1]
)[:10]
for ticker, cent in top_stocks:
    bar = "█" * int(cent * 40)
    print(f"  {ticker:<6}  {cent:.3f}  {bar}  [{sectors[ticker]}]")

# Degree centrality calculation (events)
print("\nDegree centrality — events:")
event_centrality = bipartite.degree_centrality(B, stock_nodes)
for ev in event_nodes:
    cent  = event_centrality[ev]
    label = ev.replace("\n", " ")
    bar   = "█" * int(cent * 40)
    print(f"  {label:<22}  {cent:.3f}  {bar}")

# Bipartite clustering coefficient: high = react with same group often; low = reacts independently
print("\nBipartite clustering coefficient — stocks (top 10):")
clustering = bipartite.clustering(B, nodes=stock_nodes)
top_cluster = sorted(
    [(n, c) for n, c in clustering.items()],
    key=lambda x: -x[1]
)[:10]
for ticker, clust in top_cluster:
    bar = "█" * int(clust * 40)
    print(f"  {ticker:<6}  {clust:.3f}  {bar}  [{sectors[ticker]}]")
print("  (Higher = stock shares its event reactions with other stocks -> evidence of structural interdependence)")

# Event pair overlap analysis
print("\nEvent pair overlap (shared reacting stocks):")
ev_list = list(event_nodes)
for i in range(len(ev_list)):
    for j in range(i+1, len(ev_list)):
        a, b     = ev_list[i], ev_list[j]
        set_a    = set(B.neighbors(a))
        set_b    = set(B.neighbors(b))
        shared   = set_a & set_b
        union    = set_a | set_b
        jaccard  = len(shared) / len(union) if union else 0
        la       = a.replace("\n", " ")
        lb       = b.replace("\n", " ")
        print(f"  {la:<22} ∩ {lb:<22}  {len(shared):2d} shared stocks  Jaccard: {jaccard:.2f}")

print("\n  Jaccard similarity: 0 = no overlap, 1 = identical reaction sets")
print("  High overlap between event types = interdependence; Low overlap = event-specific contagion")

# Tracking silent stocks that don't cross the established threshold
silent = [t for t in stock_nodes if B.degree(t) == 0]
print(f"\nStocks with NO significant reaction across all {len(events)} events ({len(silent)}):")
print(f"  {', '.join(sorted(silent))}")
print("  (These may represent stable, low-sensitivity market positions)")

# layout helpers
sector_order = [
    "Energy", "Materials", "Industrials", "Technology",
    "Healthcare", "Financials", "Consumer Staples", "Consumer Disc.", "Telecom"
]

# Organizes rendering layout by sector, degree constraints, and alphabetical order
tickers_sorted = sorted(
    sectors.keys(),
    key=lambda t: (sector_order.index(sectors[t]), -B.degree(t), t)
)

def bezier_curve(p0, p1, ctrl_x=0.5, n=80):
    mid_y = (p0[1] + p1[1]) / 2
    cp    = (ctrl_x, mid_y)
    t     = np.linspace(0, 1, n)
    x = (1-t)**2 * p0[0] + 2*(1-t)*t * cp[0] + t**2 * p1[0]
    y = (1-t)**2 * p0[1] + 2*(1-t)*t * cp[1] + t**2 * p1[1]
    return x, y

def draw_sector_band(ax, tickers_in_band, sector, stock_pos, sx):
    ys    = [stock_pos[t][1] for t in tickers_in_band]
    y_top = max(ys) + 0.013
    y_bot = min(ys) - 0.013
    color = sector_colors[sector]
    ax.add_patch(patches.FancyBboxPatch(
        (sx + 0.065, y_bot), 0.008, y_top - y_bot,
        boxstyle="round,pad=0.002",
        linewidth=0, facecolor=color, alpha=0.45, zorder=2
    ))
    ax.text(sx + 0.085, (y_top + y_bot) / 2, sector,
            va="center", ha="left", fontsize=6.5,
            color=color, fontweight="bold", zorder=5)

# bipartite network graph generation

event_names = list(events.keys())
n_events    = len(event_names)
n_stocks    = len(tickers_sorted)

EX, SX = 0.0, 1.0
E_YPAD = 0.08
S_YPAD = 0.02

event_pos = {}
for i, name in enumerate(event_names):
    y = 1.0 - E_YPAD - i * ((1.0 - 2*E_YPAD) / (n_events - 1))
    event_pos[name] = (EX, y)

stock_pos = {}
for i, ticker in enumerate(tickers_sorted):
    y = 1.0 - S_YPAD - i * ((1.0 - 2*S_YPAD) / (n_stocks - 1))
    stock_pos[ticker] = (SX, y)

pos = {**event_pos, **stock_pos}

fig, ax = plt.subplots(figsize=(16, 20))
fig.patch.set_facecolor(BG_COLOR)
ax.set_facecolor(BG_COLOR)
ax.set_xlim(-0.42, 1.42)
ax.set_ylim(-0.06, 1.10)
ax.axis("off")

max_weight = max(B[u][v]["weight"] for u, v in B.edges()) if B.edges() else 1

# Rendering connection links (Edges)
for u, v, data in B.edges(data=True):
    ep = pos[u] if B.nodes[u]["node_type"] == "event" else pos[v]
    sp = pos[v] if B.nodes[u]["node_type"] == "event" else pos[u]
    norm  = data["weight"] / max_weight
    alpha = 0.35 + 0.65 * norm
    lw    = 0.80 + 3.00 * norm
    color = "#FF6B6B" if data["raw"] > 0 else "#74B9FF"
    bx, by = bezier_curve(ep, sp)
    ax.plot(bx, by, color=color, lw=lw, alpha=alpha,
            zorder=1, solid_capstyle="round")

# Rendering sector container groupings
prev_sector, band_tickers = None, []
for ticker in tickers_sorted:
    s = sectors[ticker]
    if s != prev_sector:
        if prev_sector is not None:
            draw_sector_band(ax, band_tickers, prev_sector, stock_pos, SX)
        band_tickers = [ticker]
        prev_sector  = s
    else:
        band_tickers.append(ticker)
if band_tickers:
    draw_sector_band(ax, band_tickers, prev_sector, stock_pos, SX)

# Rendering stock indicator nodes
stock_degrees = {t: B.degree(t) for t in tickers_sorted}
max_deg       = max(stock_degrees.values()) if stock_degrees else 1

for ticker in tickers_sorted:
    x, y   = stock_pos[ticker]
    color  = sector_colors[sectors[ticker]]
    deg    = stock_degrees[ticker]
    radius = 0.008 + 0.007 * (deg / max_deg)
    ax.add_patch(plt.Circle((x, y), radius, color=color,
                            ec=BG_COLOR, lw=0.6, zorder=4))
    label_color = TEXT_COLOR if deg > 0 else "#aaaaaa"
    ax.text(x - 0.014, y, ticker,
            ha="right", va="center", fontsize=6.5,
            color=label_color, fontweight="bold", zorder=5)

# Rendering event indicator nodes (Scaled explicitly by tracking context metrics)
pew_max = max(meta["pew"] for meta in events.values())
for name, meta in events.items():
    x, y   = event_pos[name]
    color  = event_type_colors[meta["type"]]
    pew    = meta["pew"]
    radius = 0.030 + 0.036 * (pew / pew_max)
    for r_mult, alpha in [(2.4, 0.05), (1.7, 0.09), (1.25, 0.15)]:
        ax.add_patch(plt.Circle((x, y), radius * r_mult,
                                 color=color, alpha=alpha, zorder=2, linewidth=0))
    ax.add_patch(plt.Circle((x, y), radius, color=color,
                             ec="white", lw=1.0, zorder=6))
    ax.text(x - radius - 0.016, y + 0.008, name,
            ha="right", va="center", fontsize=9.5,
            color=TEXT_COLOR, fontweight="bold",
            linespacing=1.35, zorder=7)
    ax.text(x - radius - 0.016, y - 0.028,
            f"↳ {B.degree(name)} stocks reacted  |  Pew score: {pew}",
            ha="right", va="center", fontsize=6.5,
            color=SUBTEXT_COLOR, zorder=7)

# Legend Layout Mapping Setup
legend_y = -0.025
ax.plot([0.02, 0.055], [legend_y + 0.014]*2, color="#FF6B6B", lw=2, alpha=0.85)
ax.text(0.062, legend_y + 0.014, "Volume ▲ (spike up)",
        va="center", color="#FF6B6B", fontsize=7.5)
ax.plot([0.02, 0.055], [legend_y]*2, color="#74B9FF", lw=2, alpha=0.85)
ax.text(0.062, legend_y, "Volume ▼ (spike down)",
        va="center", color="#74B9FF", fontsize=7.5)

for i, (etype, ecolor) in enumerate(event_type_colors.items()):
    col = i % 3
    row = i // 3
    xi  = 0.30 + col * 0.22
    yi  = legend_y + 0.014 - row * 0.020
    ax.add_patch(plt.Circle((xi, yi), 0.007, color=ecolor, zorder=5))
    ax.text(xi + 0.013, yi, etype,
            va="center", color=ecolor, fontsize=7.5)

ax.text(0.76, legend_y + 0.016, "Event node size = Pew 'followed closely' %",
        va="center", color=SUBTEXT_COLOR, fontsize=7, style="italic")
ax.text(0.76, legend_y + 0.003, "Stock node size = # events reacted to",
        va="center", color=SUBTEXT_COLOR, fontsize=7, style="italic")

ax.text(0.5, 1.065, "World Events  ↔  Dow Jones Stocks",
        ha="center", va="center", fontsize=18, fontweight="bold",
        color=TEXT_COLOR, zorder=8)
ax.text(0.5, 1.047,
        f"Bipartite network  |  Q2 2011  |  Edge drawn when volume spike ≥ ±{THRESHOLD}%  |  "
        "Node size = Pew attention score (% followed closely)",
        ha="center", va="center", fontsize=8.5,
        color=SUBTEXT_COLOR, zorder=8)

plt.savefig("bipartite_final.png", dpi=150, bbox_inches="tight", facecolor=BG_COLOR)
print("\nSaved bipartite_final.png")
plt.close()

# Sankey diagram

active_tickers = [t for t in tickers_sorted if B.degree(t) > 0]
n_active       = len(active_tickers)
event_list     = list(events.keys())
n_ev           = len(event_list)

# Build dynamic flow lookup dictionary
flows = {(u if B.nodes[u]["node_type"] == "event" else v, v if B.nodes[u]["node_type"] == "event" else u): data["weight"]
         for u, v, data in B.edges(data=True)}

# Tall dynamic canvas setup context
fig2, ax2 = plt.subplots(figsize=(14, max(12, n_active * 0.38)))
fig2.patch.set_facecolor(BG_COLOR)
ax2.set_facecolor(BG_COLOR)
ax2.axis("off")

EV_X  = 0.14
STK_X = 0.68
BAR_W = 0.032

total_weight  = sum(flows.values())
USABLE_Y      = 0.88
EV_GAP_FRAC   = 0.025
STK_GAP_FRAC  = 0.008

ev_gap  = USABLE_Y * EV_GAP_FRAC
stk_gap = USABLE_Y * STK_GAP_FRAC

ev_heights  = {ev:  sum(w for (e,s),w in flows.items() if e == ev)  for ev in event_list}
stk_heights = {stk: sum(w for (e,s),w in flows.items() if s == stk) for stk in active_tickers}

ev_total_h  = sum(ev_heights.values()) if ev_heights else 1
stk_total_h = sum(stk_heights.values()) if stk_heights else 1
ev_scale    = (USABLE_Y - ev_gap  * (n_ev - 1))     / ev_total_h
stk_scale   = (USABLE_Y - stk_gap * (n_active - 1)) / stk_total_h

TOP = 0.93

# Compute Event column nodes layout positions
ev_y = {}
cursor = TOP
for ev in event_list:
    h = ev_heights[ev] * ev_scale
    ev_y[ev] = (cursor - h, cursor)
    cursor -= h + ev_gap

# Compute Stock column nodes layout positions
stk_y = {}
cursor = TOP
for stk in active_tickers:
    h = stk_heights[stk] * stk_scale
    stk_y[stk] = (cursor - h, cursor)
    cursor -= h + stk_gap

# Render side bars for events
ev_fill = {ev: ev_y[ev][0] for ev in event_list}
for ev in event_list:
    yb, yt = ev_y[ev]
    color  = event_type_colors[events[ev]["type"]]
    ax2.add_patch(patches.Rectangle((EV_X, yb), BAR_W, yt - yb, facecolor=color, alpha=0.9, zorder=3, linewidth=0))
    ax2.text(EV_X - 0.012, (yb + yt) / 2, ev.replace("\n", " "), ha="right", va="center", fontsize=9, color=TEXT_COLOR, fontweight="bold")

# Render side bars for stocks
stk_fill = {stk: stk_y[stk][0] for stk in active_tickers}
for stk in active_tickers:
    yb, yt = stk_y[stk]
    color  = sector_colors[sectors[stk]]
    ax2.add_patch(patches.Rectangle((STK_X, yb), BAR_W, yt - yb, facecolor=color, alpha=0.9, zorder=3, linewidth=0))
    ax2.text(STK_X + BAR_W + 0.008, (yb + yt) / 2, stk, ha="left", va="center", fontsize=7.5, color=TEXT_COLOR, fontweight="bold")

# Draw Ribbons helper function implementation
def draw_ribbon(ax, x0, x1, y0b, y0t, y1b, y1t, color, alpha=0.22):
    cx = (x0 + x1) / 2
    verts = [(x0, y0b), (cx, y0b), (cx, y1b), (x1, y1b), (x1, y1t), (cx, y1t), (cx, y0t), (x0, y0t), (x0, y0b)]
    codes = [Path.MOVETO, Path.CURVE4, Path.CURVE4, Path.CURVE4, Path.LINETO, Path.CURVE4, Path.CURVE4, Path.CURVE4, Path.CLOSEPOLY]
    ax.add_patch(patches.PathPatch(Path(verts, codes), facecolor=color, alpha=alpha, lw=0, zorder=1))

for ev in event_list:
    for stk in active_tickers:
        if (ev, stk) not in flows:
            continue
        w     = flows[(ev, stk)]
        h_ev  = w * ev_scale
        h_stk = w * stk_scale
        color = event_type_colors[events[ev]["type"]]

        y0b = ev_fill[ev];   y0t = y0b + h_ev;  ev_fill[ev]  += h_ev
        y1b = stk_fill[stk]; y1t = y1b + h_stk; stk_fill[stk]+= h_stk

        draw_ribbon(ax2, EV_X + BAR_W, STK_X, y0b, y0t, y1b, y1t, color=color, alpha=0.22)

# Sector Label Legend definitions
unique_sectors = list(dict.fromkeys(sectors[t] for t in active_tickers))
legend_x_start = STK_X + BAR_W + 0.07
for i, sec in enumerate(unique_sectors):
    col = i % 2
    row = i // 2
    xi  = legend_x_start + col * 0.11
    yi  = TOP - row * 0.045
    ax2.add_patch(patches.Rectangle((xi, yi - 0.012), 0.018, 0.024, facecolor=sector_colors[sec], alpha=0.9, linewidth=0))
    ax2.text(xi + 0.022, yi, sec, va="center", fontsize=6.5, color=TEXT_COLOR)

ax2.set_xlim(0.0, 1.08)
ax2.set_ylim(TOP - USABLE_Y - 0.04, TOP + 0.06)
ax2.text(0.42, TOP + 0.04, "Event → Stock Volume Flow  |  Q2 2011  |  Ribbon width = volume spike magnitude",
         ha="center", va="center", fontsize=11, fontweight="bold", color=TEXT_COLOR)

plt.savefig("sankey_final.png", dpi=150, bbox_inches="tight", facecolor=BG_COLOR)
print("Saved sankey_final.png")
plt.close()

# Pew public attention timeline

weeks = [
    {"label": "Apr 1–3",       "s1": "Japan disaster",        "s2": "Economy"},
    {"label": "Apr 7–10",      "s1": "Japan disaster",        "s2": "Gov't shutdown fight"},
    {"label": "Apr 14–17",     "s1": "Japan disaster",        "s2": "Gas and oil prices"},
    {"label": "Apr 21–25",     "s1": "Economy",               "s2": "Deadly storms"},
    {"label": "Apr 28–\nMay 1", "s1": "Deadly storms",         "s2": "Gas and oil prices"},
    {"label": "May 5–8",       "s1": "Death of\nbin Laden",   "s2": "Tornadoes and floods"},
    {"label": "May 12–15",     "s1": "Death of\nbin Laden",   "s2": "Miss. River flooding"},
    {"label": "May 19–22",     "s1": "Death of\nbin Laden",   "s2": "Miss. River flooding"},
    {"label": "May 26–29",     "s1": "Midwest tornadoes",     "s2": "Federal budget deficit"},
    {"label": "Jun 2–5",       "s1": "Economy",               "s2": "Afghanistan"},
    {"label": "Jun 9–12",      "s1": "Economy",               "s2": "Weiner scandal"},
    {"label": "Jun 16–19",     "s1": "Economy",               "s2": "Weiner resigns"},
    {"label": "Jun 23–26",     "s1": "Economy",               "s2": "Troops in Afghanistan"},
]

story_colors = {
    "Japan disaster":           "#4FC3F7",
    "Economy":                  "#FFB74D",
    "Gov't shutdown fight":     "#66BB6A",
    "Gas and oil prices":       "#EF5350",
    "Deadly storms":            "#CE93D8",
    "Death of\nbin Laden":      "#EF5350",
    "Tornadoes and floods":     "#CE93D8",
    "Miss. River flooding":     "#CE93D8",
    "Midwest tornadoes":        "#CE93D8",
    "Federal budget deficit":   "#FFB74D",
    "Afghanistan":              "#A1887F",
    "Weiner scandal":           "#90A4AE",
    "Weiner resigns":           "#90A4AE",
    "Troops in Afghanistan":    "#A1887F",
}

n_weeks  = len(weeks)
week_xs  = np.linspace(0.05, 0.95, n_weeks)
ROW1_Y   = 0.70
ROW2_Y   = 0.40
BOX_H    = 0.16

fig3, ax3 = plt.subplots(figsize=(18, 7))
fig3.patch.set_facecolor(BG_COLOR)
ax3.set_facecolor(BG_COLOR)
ax3.axis("off")

# Fixed the cut-off labels, transformation, gridlines and file saving
ax3.text(0.01, ROW1_Y, "Top News Story\n(Pew)", ha="right", va="center", fontsize=10, color=TEXT_COLOR, fontweight="bold", transform=ax3.transAxes)
ax3.text(0.01, ROW2_Y, "Second News Story\n(Pew)", ha="right", va="center", fontsize=9, color=SUBTEXT_COLOR, fontweight="bold", transform=ax3.transAxes)

for i, wk in enumerate(weeks):
    x     = week_xs[i]
    box_w = (week_xs[1] - week_xs[0]) * 0.88

    # Primary Row Rendering
    c1 = story_colors.get(wk["s1"], "#cccccc")
    ax3.add_patch(patches.FancyBboxPatch((x - box_w/2, ROW1_Y - BOX_H/2), box_w, BOX_H, boxstyle="round,pad=0.005", facecolor=c1, alpha=0.85, linewidth=0))
    ax3.text(x, ROW1_Y, wk["s1"], ha="center", va="center", fontsize=7.5, color="white", fontweight="bold", linespacing=1.3)

    # Secondary Row Rendering
    c2 = story_colors.get(wk["s2"], "#cccccc")
    ax3.add_patch(patches.FancyBboxPatch((x - box_w/2, ROW2_Y - BOX_H/2), box_w, BOX_H, boxstyle="round,pad=0.005", facecolor=c2, alpha=0.55, linewidth=0))
    ax3.text(x, ROW2_Y, wk["s2"], ha="center", va="center", fontsize=7, color=TEXT_COLOR, fontweight="bold", linespacing=1.3)

    # Date Label Axis Rendering
    ax3.text(x, 0.18, wk["label"], ha="center", va="top", fontsize=8, color=TEXT_COLOR, fontweight="bold")

ax3.text(0.5, 0.94, "Pew News Interest Index Public Attention Timeline", ha="center", va="center", fontsize=14, fontweight="bold", color=TEXT_COLOR, transform=ax3.transAxes)
plt.savefig("pew_timeline.png", dpi=150, bbox_inches="tight", facecolor=BG_COLOR)
print("Saved pew_timeline.png")
plt.close()

print("\nAll tasks completed successfully.")

BIPARTITE NETWORK SUMMARY
Threshold : ±15.0%
Nodes     : 35 (5 events + 30 stocks)
Edges     : 88

──────────────────────────────────────────────────
Event : Japan Disaster
  Reacting stocks (16): BAC, CAT, DD, DIS, GE, HPQ, IBM, JNJ, KO, KRFT, MCD, MRK, PFE, T, TRV, VZ
    BAC     ▲ +15.1%   [Financials]
    CAT     ▲ +29.4%   [Industrials]
    DD      ▼ -17.8%   [Materials]
    DIS     ▼ -22.4%   [Consumer Disc.]
    GE      ▲ +16.2%   [Industrials]
    HPQ     ▼ -23.1%   [Technology]
    IBM     ▲ +44.4%   [Technology]
    JNJ     ▼ -15.3%   [Healthcare]
    KO      ▼ -27.7%   [Consumer Staples]
    KRFT    ▼ -24.4%   [Consumer Staples]
    MCD     ▼ -15.1%   [Consumer Staples]
    MRK     ▼ -28.7%   [Healthcare]
    PFE     ▼ -24.3%   [Healthcare]
    T       ▲ +17.2%   [Telecom]
    TRV     ▼ -21.2%   [Financials]
    VZ      ▼ -17.4%   [Telecom]
──────────────────────────────────────────────────
Event : Gas & Oil Prices
  Reacting stocks (18): AA, BAC, CSCO, CVX, DD, HPQ, IBM, IN